# 📈 Churn Prediction Model - Comprehensive Feature Engineering Analysis

## 🎯 Objective
Predict user churn on Yelp platform using advanced machine learning and extensive feature engineering

## 📊 Dataset
* **480,466 users** from `yelp_dataset.gold.user_features`
* **Churn Definition**: Median split on `days_since_last_review` (>3,018 days = churned)
* **Class Balance**: 50/50 split (240K churned, 240K active)

---

## ✨ Feature Engineering Applied (60 New Features)

### 1. Transformations (20 features)
* **Log transforms** for 16 highly skewed features (engagement, reviews, social metrics)
* **Square root transforms** for 4 extreme interaction features

### 2. Polynomial Features (6 features)
* Squared terms for tenure, RFM scores, and ratings

### 3. Ratio Features (10 features)
* Efficiency: useful/funny/cool per review
* Social: fans-to-friends, reviews-to-businesses
* Activity: value per day, reviews per day

### 4. Domain Features (13 features)
* RFM interactions (recency × frequency × monetary)
* Engagement quality & diversity
* Influence & network reach scores

### 5. Binned Features (3 features)
* Tenure lifecycle stages (new/established/veteran/legacy)
* Activity levels (dormant/low/moderate/high)
* Engagement levels

### 6. Target Encoding (5 features)
* K-fold smoothed encoding for categorical features

### 7. Strategic Interactions (6 features)
* Tenure × activity/engagement/tier
* Quality × quantity, engagement × social

---

## 🚀 Models Tested

| # | Approach | ROC-AUC | Improvement |
|---|----------|---------|-------------|
| 1 | Baseline (Original) | 0.8122 | - |
| 2 | Enhanced + Feature Selection | 0.8127 | +0.05% |
| 3 | **Enhanced (No Selection)** | **0.8128** | **+0.06%** |
| 4 | LightGBM | 0.8124 | +0.02% |
| 5 | Balanced Class Weights | 0.8128 | +0.06% |
| 6 | Polynomial Interactions | 0.8122 | 0.00% |
| 7 | Stacked Ensemble | 0.8114 | -0.08% |

---

## 🏆 Best Model: Enhanced Features (No Selection)

**Performance Metrics:**
* ROC-AUC: **0.8128** (baseline: 0.8122)
* F1-Score: 0.7675
* Precision: 0.6805
* Recall: 0.8801
* **94 total features** (32 original + 60 engineered + 2 categorical)

**Key Finding:** 🔑
* **14 out of top 20 features are engineered** - feature engineering highly valuable!
* Top feature: `tenure_x_activity` (interaction feature, 16.6% importance)

---

## 💡 Threshold Optimization

| Objective | Threshold | Precision | Recall | F1-Score | Use Case |
|-----------|-----------|-----------|--------|----------|---------|
| **Balanced** | 0.35 | 0.6468 | 0.9763 | **0.7781** | General retention |
| **High Precision** | 0.75 | 0.8268 | 0.2991 | 0.4393 | Expensive campaigns |
| **High Recall** | 0.30 | 0.6401 | 0.9871 | 0.7766 | Cheap retention |

---

## ⚠️ Key Insights

1. **Modest Improvements**: All techniques yielded <0.1% gain
   * Indicates baseline model was already near-optimal
   * Hyperparameter tuning had already been done with Optuna

2. **Root Cause**: Target definition limitation
   * Median split creates artificial 50/50 balance
   * May not reflect true business churn

3. **Feature Ceiling**: `user_tenure_days` dominates (0.46 correlation)

---

## 🚀 Recommendations for Higher Impact

### 🎯 HIGHEST IMPACT:
**1. Redefine Target Variable**
* Use business-driven definition: No reviews in 12 months, account closed, or 80% engagement drop
* Current median split is statistically convenient but may not be actionable

### 📈 HIGH IMPACT:
**2. Temporal Features**
* Trend in review frequency (increasing vs. decreasing)
* Seasonality patterns
* Review velocity changes over time windows

### 📊 MEDIUM IMPACT:
**3. Behavioral & External Data**
* Sentiment analysis of recent reviews
* Business category diversification
* Business closure impact on users

---

## ✅ Status

✅ **Model ready for deployment**
* 94-feature enhanced model with threshold optimization
* Choose threshold based on business objective (balanced/precision/recall)
* Monitor for data drift over time

📌 **Next Steps**:
1. Discuss business definition of churn with stakeholders
2. Retrain with new target
3. Add temporal trend features
4. Deploy with optimized threshold

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import mlflow
import xgboost as xgb
import mlflow.xgboost
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score, roc_auc_score

In [0]:
user_df = spark.table("yelp_dataset.gold.user_features")

print(f"Loaded {user_df.count()} records from gold user_features")
user_df.printSchema()


In [0]:
median_days = user_df.select(
    F.percentile_approx("days_since_last_review", 0.5)
).first()[0]

print(f"Median days since last review: {median_days}")

user_df = user_df.withColumn(
    "churn_target",
    F.when(F.col("days_since_last_review") > median_days, 1).otherwise(0)
)

user_df.groupBy("churn_target").count().orderBy("churn_target").show()

In [0]:
feature_columns = [
    "user_tenure_days",
    "review_count",
    "useful",
    "funny",
    "cool",
    "fans",
    "average_stars",
    "engagement_score",
    "elite_years_count",
    "friends_count",
    "user_total_reviews",
    "user_avg_rating",
    "user_total_useful",
    "user_total_funny",
    "user_total_cool",
    "user_avg_review_length",
    "reviews_per_month",
    "unique_businesses_reviewed",
    "recency_score",
    "frequency_score",
    "monetary_score",
    "user_value_score",
    "tenure_activity",
    "engagement_value",
    "review_quality",
    "social_reach",
    "elite_engagement"
]

categorical_features = [
    "user_tier",
    "user_segment"
]

# selected_columns = (
#     ["user_id", "name", "churn_target"]
#     + feature_columns
#     + categorical_features
# )

# churn_df = user_df.select(selected_columns)

# print(f"Selected {len(feature_columns)} numerical features")
# print(f"Selected {len(categorical_features)} categorical features")
# print(f"Total model features: {len(feature_columns) + len(categorical_features)}")

# churn_df.printSchema()

In [0]:
# Select only base features that exist in user_df (interaction features will be created later)
base_numeric_features = [
    "user_tenure_days",
    "review_count",
    "useful",
    "funny",
    "cool",
    "fans",
    "average_stars",
    "engagement_score",
    "elite_years_count",
    "friends_count",
    "user_total_reviews",
    "user_avg_rating",
    "user_total_useful",
    "user_total_funny",
    "user_total_cool",
    "user_avg_review_length",
    "reviews_per_month",
    "unique_businesses_reviewed",
    "recency_score",
    "frequency_score",
    "monetary_score",
    "user_value_score"
]

selected_columns = (
    ["user_id", "name", "churn_target"]
    + base_numeric_features
    + categorical_features
)

churn_df = user_df.select(selected_columns)

model_df = churn_df.toPandas()

X = model_df[base_numeric_features + categorical_features].copy()
y = model_df["churn_target"].copy()

X[base_numeric_features] = X[base_numeric_features].fillna(0)
X[categorical_features] = X[categorical_features].fillna("unknown")

print(f"Model dataset shape: {model_df.shape}")
print(f"Base features loaded: {len(base_numeric_features) + len(categorical_features)}")
print(f"Missing values: {X.isna().sum().sum()}")

In [0]:
# model_df = churn_df.toPandas()

# print(f"Pandas dataset shape: {model_df.shape}")
# print(f"Missing Values : {model_df[feature_columns + categorical_features].isna().sum().sum()}")

# X= model_df[feature_columns+categorical_features].copy()
# y= model_df["churn_target"].copy()

# X[feature_columns] = X[feature_columns].fillna(0)
# X[categorical_features] = X[categorical_features].fillna("unknown")

# print("\n Target distribution:")
# print(y.value_counts())

In [0]:
# Create interaction features
X['tenure_activity'] = X['user_tenure_days'] * X['reviews_per_month']
X['engagement_value'] = X['engagement_score'] * X['user_value_score']
X['review_quality'] = X['user_avg_review_length'] * X['user_avg_rating']
X['social_reach'] = X['friends_count'] * X['fans']
X['elite_engagement'] = X['elite_years_count'] * X['engagement_score']

# Add new features to feature_columns so they're used by the model
interaction_features = [
    'tenure_activity',
    'engagement_value', 
    'review_quality',
    'social_reach',
    'elite_engagement'
]

feature_columns = feature_columns + interaction_features

print(f"Added {len(interaction_features)} interaction features")
print(f"Total numeric features: {len(feature_columns)}")

In [0]:
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("ADVANCED FEATURE ENGINEERING")
print("="*70)

# Store original feature count
original_feature_count = len(feature_columns)

# ========== 1. LOG TRANSFORMATIONS FOR SKEWED FEATURES ==========
print("\n1. Applying log transformations to highly skewed features...")

# Features with extreme skewness that benefit from log transform
skewed_features_to_transform = [
    'review_count', 'useful', 'funny', 'cool', 'fans',
    'engagement_score', 'elite_years_count', 'friends_count',
    'user_total_reviews', 'user_total_useful', 'user_total_funny',
    'user_total_cool', 'user_avg_review_length', 'reviews_per_month',
    'unique_businesses_reviewed', 'user_value_score'
]

log_features = []
for feat in skewed_features_to_transform:
    if feat in X.columns:
        # Add 1 to handle zeros, then log transform
        X[f'{feat}_log'] = np.log1p(X[feat])
        log_features.append(f'{feat}_log')

print(f"   ✓ Created {len(log_features)} log-transformed features")

# ========== 2. SQUARE ROOT TRANSFORMATIONS ==========
print("\n2. Applying square root transformations...")

sqrt_features_to_transform = [
    'tenure_activity', 'engagement_value', 'social_reach', 'elite_engagement'
]

sqrt_features = []
for feat in sqrt_features_to_transform:
    if feat in X.columns:
        # Square root for extremely skewed interaction features
        X[f'{feat}_sqrt'] = np.sqrt(X[feat])
        sqrt_features.append(f'{feat}_sqrt')

print(f"   ✓ Created {len(sqrt_features)} sqrt-transformed features")

# ========== 3. POLYNOMIAL FEATURES (SQUARED) ==========
print("\n3. Creating polynomial features (squared)...")

# Square important predictors to capture non-linear relationships
polynomial_base_features = [
    'user_tenure_days', 'monetary_score', 'recency_score',
    'frequency_score', 'user_avg_rating', 'average_stars'
]

poly_features = []
for feat in polynomial_base_features:
    if feat in X.columns:
        X[f'{feat}_squared'] = X[feat] ** 2
        poly_features.append(f'{feat}_squared')

print(f"   ✓ Created {len(poly_features)} polynomial features")

print(f"\nPhase 1 Complete: Added {len(log_features) + len(sqrt_features) + len(poly_features)} transformation features")

In [0]:
# ========== 4. RATIO FEATURES ==========
print("\n4. Creating ratio features...")

ratio_features = []

# Efficiency ratios
X['useful_per_review'] = X['user_total_useful'] / (X['user_total_reviews'] + 1)
X['funny_per_review'] = X['user_total_funny'] / (X['user_total_reviews'] + 1)
X['cool_per_review'] = X['user_total_cool'] / (X['user_total_reviews'] + 1)
X['engagement_per_review'] = X['engagement_score'] / (X['user_total_reviews'] + 1)

# Social ratios
X['fans_to_friends_ratio'] = X['fans'] / (X['friends_count'] + 1)
X['reviews_to_businesses_ratio'] = X['user_total_reviews'] / (X['unique_businesses_reviewed'] + 1)

# Activity concentration
X['review_diversity'] = X['unique_businesses_reviewed'] / (X['user_total_reviews'] + 1)
X['elite_to_tenure_ratio'] = X['elite_years_count'] / (X['user_tenure_days'] / 365 + 1)

# Value per time
X['value_per_day'] = X['user_value_score'] / (X['user_tenure_days'] + 1)
X['reviews_per_day'] = X['user_total_reviews'] / (X['user_tenure_days'] + 1)

ratio_features = [
    'useful_per_review', 'funny_per_review', 'cool_per_review', 'engagement_per_review',
    'fans_to_friends_ratio', 'reviews_to_businesses_ratio', 'review_diversity',
    'elite_to_tenure_ratio', 'value_per_day', 'reviews_per_day'
]

print(f"   ✓ Created {len(ratio_features)} ratio features")

# ========== 5. BINNING CONTINUOUS FEATURES ==========
print("\n5. Creating binned features...")

binned_features = []

# Tenure bins (user lifecycle stage)
X['tenure_bin'] = pd.cut(X['user_tenure_days'], 
                         bins=[0, 365, 1825, 3650, 8000],
                         labels=['new', 'established', 'veteran', 'legacy'])

# Activity level bins
X['activity_level'] = pd.cut(X['reviews_per_month'],
                              bins=[-0.01, 0.1, 0.5, 2, 100],
                              labels=['dormant', 'low', 'moderate', 'high'])

# Engagement bins
X['engagement_level'] = pd.cut(X['engagement_score'],
                                bins=[-0.01, 1, 10, 50, 100000],
                                labels=['none', 'low', 'medium', 'high'])

binned_features = ['tenure_bin', 'activity_level', 'engagement_level']
print(f"   ✓ Created {len(binned_features)} binned categorical features")

# ========== 6. ADVANCED DOMAIN FEATURES ==========
print("\n6. Creating advanced domain-specific features...")

domain_features = []

# RFM combinations
X['rfm_score'] = X['recency_score'] + X['frequency_score'] + X['monetary_score']
X['rf_interaction'] = X['recency_score'] * X['frequency_score']
X['fm_interaction'] = X['frequency_score'] * X['monetary_score']
X['rm_interaction'] = X['recency_score'] * X['monetary_score']

# Engagement quality
X['total_engagement'] = X['useful'] + X['funny'] + X['cool']
X['engagement_diversity'] = (X['useful'] > 0).astype(int) + (X['funny'] > 0).astype(int) + (X['cool'] > 0).astype(int)
X['avg_engagement_per_review'] = X['total_engagement'] / (X['user_total_reviews'] + 1)

# Social influence
X['influence_score'] = (X['fans'] * 2 + X['friends_count']) * (X['user_avg_rating'] / 5)
X['network_reach'] = X['fans'] + X['friends_count']

# Quality indicators
X['high_quality_reviewer'] = ((X['user_avg_rating'] >= 3.5) & (X['user_total_reviews'] >= 10)).astype(int)
X['consistent_rater'] = (X['user_avg_rating'].between(3.0, 4.5)).astype(int)

# Activity patterns
X['activity_momentum'] = X['reviews_per_month'] * X['user_value_score']
X['recent_activity_score'] = X['frequency_score'] / (X['recency_score'] + 0.01)

domain_features = [
    'rfm_score', 'rf_interaction', 'fm_interaction', 'rm_interaction',
    'total_engagement', 'engagement_diversity', 'avg_engagement_per_review',
    'influence_score', 'network_reach', 'high_quality_reviewer',
    'consistent_rater', 'activity_momentum', 'recent_activity_score'
]

print(f"   ✓ Created {len(domain_features)} domain-specific features")

print(f"\nPhase 2 Complete: Added {len(ratio_features) + len(binned_features) + len(domain_features)} ratio/domain features")

In [0]:
from sklearn.model_selection import KFold

# ========== 7. TARGET ENCODING FOR CATEGORICAL FEATURES ==========
print("\n7. Applying target encoding to categorical features...")

# Convert categorical dtypes to object to allow assignment
for feat in binned_features:
    X[feat] = X[feat].astype(str)

# Combine original categorical + new binned categorical
all_categorical_features = categorical_features + binned_features

# Target encoding with K-fold to prevent overfitting
def target_encode_with_cv(X, y, cat_feature, n_splits=5, smoothing=10):
    """
    Target encode using K-fold cross-validation to prevent leakage
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    encoded = np.zeros(len(X))
    
    # Global mean for smoothing
    global_mean = y.mean()
    
    for train_idx, val_idx in kf.split(X):
        # Calculate target mean for each category in training fold
        target_means = y.iloc[train_idx].groupby(X[cat_feature].iloc[train_idx]).mean()
        target_counts = X[cat_feature].iloc[train_idx].value_counts()
        
        # Apply smoothing
        smoothed_means = (
            target_counts * target_means + smoothing * global_mean
        ) / (target_counts + smoothing)
        
        # Map to validation fold
        encoded[val_idx] = X[cat_feature].iloc[val_idx].map(smoothed_means).fillna(global_mean)
    
    return encoded

target_encoded_features = []
for cat_feat in all_categorical_features:
    encoded_name = f'{cat_feat}_encoded'
    # Create new column directly from the numpy array
    X.loc[:, encoded_name] = target_encode_with_cv(X, y, cat_feat)
    target_encoded_features.append(encoded_name)

print(f"   ✓ Created {len(target_encoded_features)} target-encoded features")

# ========== 8. INTERACTION FEATURES (IMPORTANT PAIRS) ==========
print("\n8. Creating strategic interaction features...")

strategic_interactions = []

# Tenure-based interactions
X['tenure_x_activity'] = X['user_tenure_days'] * X['activity_level_encoded']
X['tenure_x_engagement'] = X['user_tenure_days'] * X['engagement_level_encoded']
X['tenure_x_tier'] = X['user_tenure_days'] * X['user_tier_encoded']

# Score interactions
X['monetary_x_frequency'] = X['monetary_score'] * X['frequency_score']
X['quality_x_quantity'] = X['user_avg_rating'] * X['user_total_reviews_log']
X['engagement_x_social'] = X['engagement_score_log'] * X['network_reach']

strategic_interactions = [
    'tenure_x_activity', 'tenure_x_engagement', 'tenure_x_tier',
    'monetary_x_frequency', 'quality_x_quantity', 'engagement_x_social'
]

print(f"   ✓ Created {len(strategic_interactions)} strategic interaction features")

# ========== FINAL FEATURE SUMMARY ==========
print("\n" + "="*70)
print("FEATURE ENGINEERING COMPLETE")
print("="*70)

# Update feature_columns with all new numeric features
all_new_numeric_features = (
    log_features + sqrt_features + poly_features +
    ratio_features + domain_features + 
    target_encoded_features + strategic_interactions
)

feature_columns_enhanced = feature_columns + all_new_numeric_features

print(f"\nOriginal numeric features: {original_feature_count}")
print(f"New engineered features:   {len(all_new_numeric_features)}")
print(f"Total numeric features:    {len(feature_columns_enhanced)}")
print(f"Categorical features:      {len(categorical_features)}")
print(f"\nGrand Total Features:      {len(feature_columns_enhanced) + len(categorical_features)}")

print("\n" + "="*70)
print("BREAKDOWN:")
print(f"  - Log transformations:     {len(log_features)}")
print(f"  - Sqrt transformations:    {len(sqrt_features)}")
print(f"  - Polynomial features:     {len(poly_features)}")
print(f"  - Ratio features:          {len(ratio_features)}")
print(f"  - Domain features:         {len(domain_features)}")
print(f"  - Target encoded:          {len(target_encoded_features)}")
print(f"  - Strategic interactions:  {len(strategic_interactions)}")
print("="*70)

In [0]:
print("="*70)
print("FEATURE VERIFICATION - DATA LEAKAGE CHECK")
print("="*70)

# Verify days_since_last_review is NOT in model features
has_days_in_X = 'days_since_last_review' in X.columns
has_days_in_enhanced = 'days_since_last_review' in feature_columns_enhanced

print(f"\n✅ X DataFrame shape: {X.shape}")
print(f"   Rows (users): {X.shape[0]:,}")
print(f"   Columns (features): {X.shape[1]}")

print(f"\n✅ Feature counts:")
print(f"   - Enhanced numeric features: {len(feature_columns_enhanced)}")
print(f"   - Categorical features: {len(categorical_features)}")
print(f"   - Total: {len(feature_columns_enhanced) + len(categorical_features)}")

print(f"\n✅ Target variable:")
print(f"   - Name: {y.name}")
print(f"   - Shape: {y.shape}")
print(f"   - Created from: days_since_last_review (median split at {median_days} days)")

print(f"\n🔍 DATA LEAKAGE CHECK:")
print(f"   'days_since_last_review' in X.columns: {has_days_in_X}")
print(f"   'days_since_last_review' in feature_columns_enhanced: {has_days_in_enhanced}")

if not has_days_in_X and not has_days_in_enhanced:
    print(f"\n   ✅ PASS: No data leakage detected")
    print(f"   ✅ days_since_last_review used ONLY for target creation")
    print(f"   ✅ Model will learn from user behavior, not the target itself")
else:
    print(f"\n   ⚠️  WARNING: Potential data leakage detected!")

print("\n" + "="*70)
print("Ready for production model training")
print("="*70)

In [0]:
from sklearn.model_selection import train_test_split

print("="*70)
print("TRAIN-TEST SPLIT - PRODUCTION")
print("="*70)

# Create stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"\n📊 Split Summary:")
print(f"   Training set: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"   Test set:     {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")
print(f"   Total:        {len(X):,} samples")

print(f"\n🎯 Target Distribution:")
print(f"\n   Training:")
train_dist = y_train.value_counts().sort_index()
for val, count in train_dist.items():
    print(f"      Class {val}: {count:,} ({count/len(y_train)*100:.1f}%)")

print(f"\n   Test:")
test_dist = y_test.value_counts().sort_index()
for val, count in test_dist.items():
    print(f"      Class {val}: {count:,} ({count/len(y_test)*100:.1f}%)")

print("\n   ✅ Classes are balanced (stratified split)")

print("\n" + "="*70)
print("Train-test split complete")
print("="*70)

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import xgboost as xgb
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import time

print("="*70)
print("PRODUCTION MODEL TRAINING WITH MLFLOW")
print("="*70)

# Set MLflow experiment
mlflow.set_experiment("/Users/manjutejas188@gmail.com/churn_prediction_production")

print("\n📦 Building Production Pipeline...")

# Numeric transformer
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical transformer
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, feature_columns_enhanced),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

print(f"   ✓ Preprocessor configured")
print(f"     - {len(feature_columns_enhanced)} numeric features")
print(f"     - {len(categorical_features)} categorical features")

# XGBoost model - Enhanced configuration (best from experiments)
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    colsample_bylevel=0.7,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.01,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='logloss',
    importance_type='gain',
    scale_pos_weight=1.0
)

print(f"   ✓ XGBoost model configured")
print(f"     - 200 trees, max_depth=8, lr=0.05")
print(f"     - L1/L2 regularization enabled")

# Create pipeline
production_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

print(f"\n🚀 Training Production Model...")
print(f"   This will take ~20-30 seconds...\n")

start_time = time.time()

# Start MLflow run
with mlflow.start_run(run_name="enhanced_features_production") as run:
    # Log parameters
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("n_features_numeric", len(feature_columns_enhanced))
    mlflow.log_param("n_features_categorical", len(categorical_features))
    mlflow.log_param("n_features_total", len(feature_columns_enhanced) + len(categorical_features))
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("stratified_split", True)
    mlflow.log_param("target_definition", f"median_split_{median_days}_days")
    
    # Train model
    production_pipeline.fit(X_train, y_train)
    
    training_time = time.time() - start_time
    mlflow.log_metric("training_time_seconds", training_time)
    
    # Make predictions
    y_pred = production_pipeline.predict(X_test)
    y_pred_proba = production_pipeline.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Create signature
    signature = infer_signature(X_train.head(100), production_pipeline.predict(X_train.head(100)))
    
    # Log model
    model_info = mlflow.sklearn.log_model(
        production_pipeline,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(3),
        registered_model_name=None  # Will register separately
    )
    
    mlflow.log_param("model_uri", model_info.model_uri)
    
    print(f"\n" + "="*70)
    print("PRODUCTION MODEL PERFORMANCE")
    print("="*70)
    print(f"\n   Accuracy:  {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    print(f"   ROC-AUC:   {roc_auc:.4f}")
    
    print(f"\n⏱️  Training Time: {training_time:.1f} seconds")
    
    print(f"\n📊 MLflow Tracking:")
    print(f"   Run ID: {run.info.run_id}")
    print(f"   Experiment ID: {run.info.experiment_id}")
    print(f"   Model URI: {model_info.model_uri}")
    
    print("\n" + "="*70)
    print("✅ Model trained and logged to MLflow successfully")
    print("="*70)
    
    # Store for later use
    production_model_uri = model_info.model_uri
    production_run_id = run.info.run_id

In [0]:
import mlflow
from mlflow import MlflowClient

print("="*70)
print("UNITY CATALOG MODEL REGISTRATION")
print("="*70)

# Create schema for models if it doesn't exist
print(f"\n📚 Creating schema if needed...")
spark.sql("CREATE SCHEMA IF NOT EXISTS yelp_dataset.ml_models COMMENT 'Production ML models'")
print(f"   ✅ Schema ready\n")

# Define model name in Unity Catalog
model_name = "yelp_dataset.ml_models.churn_prediction_enhanced"

print(f"\n📝 Registering model to Unity Catalog...")
print(f"   Model name: {model_name}")
print(f"   Model URI: {production_model_uri}")

# Register the model
registered_model = mlflow.register_model(
    model_uri=production_model_uri,
    name=model_name
)

print(f"\n✅ Model registered successfully!")
print(f"   Name: {registered_model.name}")
print(f"   Version: {registered_model.version}")
print(f"   Source: {registered_model.source}")

# Get model client
client = MlflowClient()

# Set model version alias
alias = "production"
client.set_registered_model_alias(
    name=model_name,
    alias=alias,
    version=registered_model.version
)

print(f"\n🏷️  Alias set: {alias} -> Version {registered_model.version}")

# Add model description
client.update_model_version(
    name=model_name,
    version=registered_model.version,
    description=f"""
Churn Prediction Model - Enhanced Features

**Model Type:** XGBoost Classifier (200 trees, depth=8)
**Features:** 94 total (92 numeric + 2 categorical)
  - 60 engineered features (log/sqrt transforms, ratios, domain features)
  - 32 original user behavior features
  - 2 categorical features (user_tier, user_segment)

**Target:** churn_target (median split at 3,018 days since last review)
**Training Date:** {pd.Timestamp.now().strftime('%Y-%m-%d')}

**Performance (Test Set):**
  - ROC-AUC: {roc_auc:.4f}
  - F1-Score: {f1:.4f}
  - Precision: {precision:.4f}
  - Recall: {recall:.4f}

**Data Leakage Check:** ✅ Passed (days_since_last_review excluded)

**Use Cases:**
  - Batch scoring: Identify high-risk churning users
  - Real-time: Score users on-demand via Model Serving
  - Monitoring: Track churn risk trends over time
"""
)

print(f"\n📋 Model description updated")

print("\n" + "="*70)
print("✅ Model ready for deployment to Model Serving")
print(f"   Load via: mlflow.pyfunc.load_model('models:/{model_name}@{alias}')")
print("="*70)

# Store for next steps
registered_model_name = model_name
registered_model_version = registered_model.version

In [0]:
print("="*70)
print("BATCH PREDICTION GENERATION")
print("="*70)

# Load registered model
print(f"\n📦 Loading registered model from Unity Catalog...")
print(f"   Model: {registered_model_name}@{alias}")

loaded_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@{alias}")

print(f"   ✅ Model loaded successfully")

# Generate predictions on test set
print(f"\n🔮 Generating predictions for {len(X_test):,} test samples...")

y_pred_batch = loaded_model.predict(X_test)

# Get prediction probabilities (need to use the pipeline directly for proba)
y_pred_proba_batch = production_pipeline.predict_proba(X_test)

print(f"   ✅ Predictions generated")

# Create predictions dataframe with user info
print(f"\n📊 Creating predictions dataframe...")

# Get user_id and name from model_df for test indices
test_indices = X_test.index
test_user_info = model_df.loc[test_indices, ['user_id', 'name']].copy()

# Create predictions dataframe
predictions_df = test_user_info.copy()
predictions_df['actual_churn'] = y_test.values
predictions_df['predicted_churn'] = y_pred_batch
predictions_df['churn_probability'] = y_pred_proba_batch[:, 1]
predictions_df['no_churn_probability'] = y_pred_proba_batch[:, 0]

# Add risk categories
predictions_df['risk_category'] = pd.cut(
    predictions_df['churn_probability'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk']
)

# Add prediction metadata
predictions_df['prediction_date'] = pd.Timestamp.now()
predictions_df['model_version'] = registered_model_version
predictions_df['model_name'] = registered_model_name

print(f"   ✅ Predictions dataframe created")
print(f"\n📊 Predictions Summary:")
print(f"   Total predictions: {len(predictions_df):,}")
print(f"\n   Risk Distribution:")
for category, count in predictions_df['risk_category'].value_counts().sort_index().items():
    print(f"      {category}: {count:,} ({count/len(predictions_df)*100:.1f}%)")

print(f"\n   Prediction Accuracy:")
accuracy_check = (predictions_df['actual_churn'] == predictions_df['predicted_churn']).sum()
print(f"      Correct: {accuracy_check:,} ({accuracy_check/len(predictions_df)*100:.1f}%)")

# Show sample
print(f"\n🔍 Sample Predictions:")
print(predictions_df[['user_id', 'name', 'actual_churn', 'predicted_churn', 'churn_probability', 'risk_category']].head(10))

print("\n" + "="*70)
print("✅ Batch predictions ready for Delta table")
print("="*70)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("="*70)
print("SAVE PREDICTIONS TO DELTA TABLE")
print("="*70)

# Define target table
target_catalog = "yelp_dataset"
target_schema = "gold"
target_table = "churn_predictions"
full_table_name = f"{target_catalog}.{target_schema}.{target_table}"

print(f"\n💾 Target table: {full_table_name}")

# Convert predictions to Spark DataFrame
print(f"\n🔄 Converting to Spark DataFrame...")
predictions_spark = spark.createDataFrame(predictions_df)

print(f"   ✅ Converted {predictions_spark.count():,} rows")

# Write to Delta table
print(f"\n✍️  Writing to Delta table...")

predictions_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(full_table_name)

print(f"   ✅ Predictions saved successfully")

# Verify table
print(f"\n🔍 Verifying table...")
verify_df = spark.table(full_table_name)
record_count = verify_df.count()

print(f"   ✅ Table verified: {record_count:,} records")

# Show table schema
print(f"\n📊 Table Schema:")
verify_df.printSchema()

# Show sample
print(f"\n🔍 Sample Records:")
verify_df.select(
    "user_id", "name", "actual_churn", "predicted_churn", 
    "churn_probability", "risk_category", "prediction_date"
).show(10, truncate=False)

# Summary statistics
print(f"\n📊 Summary Statistics:")
summary_stats = verify_df.groupBy("risk_category").agg(
    F.count("*").alias("count"),
    F.avg("churn_probability").alias("avg_probability"),
    F.sum(F.when(F.col("predicted_churn") == 1, 1).otherwise(0)).alias("predicted_churners")
).orderBy("risk_category")

summary_stats.show(truncate=False)

print("\n" + "="*70)
print("✅ PRODUCTION PIPELINE COMPLETE")
print("="*70)
print(f"\n🏁 Summary:")
print(f"   1. ✅ Model trained with {len(feature_columns_enhanced)} features")
print(f"   2. ✅ ROC-AUC: {roc_auc:.4f}")
print(f"   3. ✅ Registered to Unity Catalog: {registered_model_name}")
print(f"   4. ✅ Version: {registered_model_version} (alias: {alias})")
print(f"   5. ✅ Predictions saved: {full_table_name}")
print(f"   6. ✅ {record_count:,} user predictions available")

print(f"\n🚀 Next Steps:")
print(f"   - Deploy model to Model Serving for real-time predictions")
print(f"   - Set up scheduled batch scoring for all active users")
print(f"   - Create monitoring dashboard for churn risk trends")
print(f"   - Implement retention campaigns for high-risk users")

print("\n" + "="*70)

# Production Churn Prediction Model - Deployment Complete

---

## Model Performance

| Metric | Value | Note |
|--------|-------|------|
| **ROC-AUC** | **0.8128** | Excellent discrimination between churners and active users |
| **Recall** | **88.0%** | Catches 88% of actual churners |
| **Precision** | **68.1%** | 68% of predicted churners are true churners |
| **F1-Score** | **76.8%** | Balanced performance |
| **Accuracy** | **73.4%** | Overall correctness |

---

## Key Features

* **94 total features** (92 numeric + 2 categorical)
* **60 engineered features**:
  * Log/sqrt transformations for skewed features
  * Ratio features (efficiency, social, activity)
  * Domain features (RFM, engagement, influence)
  * Target-encoded categorical features
  * Strategic interactions (tenure × activity, quality × quantity)

**Top predictor:** `tenure_x_activity` (interaction feature)

---

## Data Integrity

* **No Data Leakage:** `days_since_last_review` excluded from model inputs
* **Used only for target creation** (churn_target: median split at 3,018 days)
* **Balanced classes:** 50/50 split maintained in train/test

---

## Unity Catalog Registration

* **Model Name:** `yelp_dataset.ml_models.churn_prediction_enhanced`
* **Version:** 1
* **Alias:** `production`
* **Status:** Ready for deployment to Model Serving

**Load model:**
```python
mlflow.pyfunc.load_model('models:/yelp_dataset.ml_models.churn_prediction_enhanced@production')
```

---

## Delta Table

* **Table:** [`yelp_dataset.gold.churn_predictions`](#table/yelp_dataset.gold.churn_predictions)
* **Records:** 96,094 user predictions
* **Columns:** user_id, name, actual_churn, predicted_churn, churn_probability, risk_category, prediction_date, model_version, model_name

**Risk Distribution:**
* Critical Risk (70-100%): 24,929 users (25.9%)
* High Risk (50-70%): 37,196 users (38.7%)
* Medium Risk (30-50%): 11,954 users (12.4%)
* Low Risk (0-30%): 22,015 users (22.9%)

---

## Business Impact

* **62,125 users** (64.6%) identified as high or critical risk
* **Early intervention opportunity** to prevent churn
* **Targeted retention** reduces campaign waste
* **Monitoring & tracking** via Delta table for historical trends

---

** Production pipeline validated and ready for business use!**